# Hacking Creativity Glyph Example

Notebook companion to `generate_hacking_creativity_glyph.py`.

**The dataset**: Red Bull High Performance Team's 2015 "Hacking Creativity" study, run on Vibrant Data's Mappr Platform -- a creative-style survey of ~660 people, clustered by similarity into 13 Louvain groups (`Cluster`/`Louvain0` column), with network stats (`degree`, `betweennessCentrality`, `centrality_/bridging_/diversity_Louvain0`) and ~13 Mappr-derived "creative style" dimensions (Pace, Mood, Kinetic/Cerebral/Intuitive, etc.) already computed per person.

**Why 13 glyphs, not ~660**: an earlier per-person version was built first, but checking the data directly showed the 13 derived dimension columns are **100% identical across every member of a cluster** -- they're literally what `ClusterLabel` is assembled from, not an independent per-person signal. So one glyph per person added zero information beyond the first. The real per-person variation in this dataset (raw survey text, network position) doesn't repeat that way, which is why the final design promotes *cluster-level* network statistics instead of stacking near-duplicate individuals.

**Glyph design** (one per cluster):
- Root sphere: color = pale hash of `ClusterLabel`; size = cluster member count (log-normalized across all 13 clusters).
- Up to 13 small spheres ringed around the root at fixed angular slots, one per derived dimension -- hue = dimension identity (same angle/hue on every cluster's glyph, so rings are directly comparable), brightness = hash of that cluster's value. A slot is left empty if the dimension wasn't computed for anyone in the cluster (an honest gap).
- A wire-torus halo one ring further out: color (white -> gold) and thickness = **mean degree** within the cluster.
- A gold "hub spike" raised above the root: height/size = the **highest single degree** in the cluster, so a cluster with one standout hub looks different from a uniformly peripheral one even at the same mean degree.
- Three small cube markers below the equator (a deliberately different shape from the round dimension spokes): mean `centrality_Louvain0` / `bridging_Louvain0` / `diversity_Louvain0`, brightness-scaled.
- Every node has a tag, but they're meant to be read with GlyphViz's existing **Tags: Selected Only** toggle on -- click a node to see its detail rather than having everything labeled at once.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from generate_hacking_creativity_glyph import (
    load_all_people, build_all_clusters, build_person, write_csvs,
    cluster_sort_key, PREFIX, OUTPUT_DIR,
)

## Load the dataset

Reads the source xlsx directly (path is hardcoded in `XLSX_PATH` at the top of the script) and drops the one bogus header-artifact row at index 0.

In [ ]:
df = load_all_people()

clusters = sorted(df["Cluster"].unique(), key=cluster_sort_key)
summary = (
    df.groupby("Cluster")
    .agg(n=("id", "size"), label=("ClusterLabel", "first"),
         mean_degree=("degree", "mean"), max_degree=("degree", "max"))
    .loc[clusters]
)
print(f"{len(df)} participants, {len(clusters)} clusters")
summary.round(2)

## Generate the 13 cluster-exemplar glyphs

In [ ]:
node_rows, tag_rows = build_all_clusters(df)
node_path, tag_path = write_csvs(node_rows, tag_rows, OUTPUT_DIR, PREFIX)
node_path, tag_path

## Peek at the output

In [ ]:
import pandas as pd

nodes_df = pd.read_csv(node_path)
tags_df = pd.read_csv(tag_path)

print("Nodes per branch level:")
print(nodes_df["branch_level"].value_counts().sort_index())
tags_df.head(10)

## Optional: a single real person's glyph

`build_person` is the building block the original per-person version used, kept around for inspecting one real individual (e.g. to sanity-check against the cluster exemplar they belong to). Not part of the main committed output -- this writes to a separate `_person<id>_gv_node/tag.csv` pair.

In [ ]:
person_id = 218
row = df[df["id"] == person_id].iloc[0]
p_node_rows, p_tag_rows, _, _ = build_person(row.to_dict(), person_id, (0.0, 0.0, 0.0), 1.0, 1, 1)
p_node_path, p_tag_path = write_csvs(p_node_rows, p_tag_rows, OUTPUT_DIR, f"{PREFIX}_person{person_id}")
p_node_path, p_tag_path

## Optional: render a quick preview

Uses the same headless-GL pattern as `tools/export_frame.py` (loads the CSV through the real `Viewport`, grabs a framebuffer -- no app window needed). Requires PySide6 + the `glyphviz_gl`/`glyphviz_core` packages to be importable, i.e. run this notebook's kernel from the repo root (or add the repo root to `sys.path` below).

In [ ]:
REPO_ROOT = HERE.parent.parent  # examples/Hacking_Creativity_Glyph_Example -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from PySide6.QtWidgets import QApplication

app = QApplication.instance() or QApplication(sys.argv[:1])

from glyphviz_core.csv_reader import load_node_csv
from glyphviz_gl.viewport import Viewport

nodes = load_node_csv(str(node_path))
vp = Viewport()
vp.base_scale = 3.0
vp.show_grid = True
vp.show_axes = True
vp.show_tags = False
vp.set_nodes(nodes)
vp.set_camera(azimuth=20, elevation=45, distance=140, target=(30, -30, 0))
preview_path = OUTPUT_DIR / "preview.png"
vp.export_png(str(preview_path), 1600, 1200)
preview_path

## Load in GlyphViz

File > Open Node CSV -> the `node_path` printed above (`Hacking_Creativity_Glyph_Example_gv_node.csv`). Then enable Display > **Tags: Selected Only** and click any node to read its detail.